## Detectron2 finetuning setup

This notebook trains and evaluates a Detectron2 Mask R-CNN model for tray segmentation. The paths below are resolved from the repository root so the notebook is easier to reuse on another machine.

In [ ]:
from pathlib import Path
import os
import random

import cv2
import detectron2
import matplotlib.pyplot as plt
import torch

# Advanced users: edit these paths directly in this notebook.
ROOT_PATH = Path("/home/vasakjakub/fenotypizace")
SEGMENTATION_MODEL_DIR = ROOT_PATH / "models" / "segmentation_model"
ANNOTATIONS_PATH = SEGMENTATION_MODEL_DIR / "annotations"
SEGMENTATION_OUTPUT_DIR = SEGMENTATION_MODEL_DIR / "output"

TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]

print("Repo root:", ROOT_PATH)
print("Train annotations:", ANNOTATIONS_PATH / "train" / "result.json")
print("Test annotations:", ANNOTATIONS_PATH / "test" / "result.json")
print("Output dir:", SEGMENTATION_OUTPUT_DIR)
print("torch:", TORCH_VERSION, "; cuda:", CUDA_VERSION)
print("detectron2:", detectron2.__version__)
print("CUDA available:", torch.cuda.is_available())

torch:  2.6 ; cuda:  cu124
detectron2: 0.6


In [ ]:
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.engine import DefaultPredictor
from detectron2.utils.logger import setup_logger
from detectron2.utils.visualizer import Visualizer

setup_logger()

<Logger detectron2 (DEBUG)>

## Loading the data

- The data is in COCO format. The register_coco_instances function registers the datasets. The MetadataCatalog.get and DatasetCatalog.get functions retrieve the metadata and the dataset respectively. The results are printed at the end.

In [ ]:
annotations_path = SEGMENTATION_MODEL_DIR / "annotations"

train_images = annotations_path / "train" / "images"
train_json = annotations_path / "train" / "result.json"

test_images = annotations_path / "test" / "images"
test_json = annotations_path / "test" / "result.json"

if "my_trainset" not in DatasetCatalog.list():
    register_coco_instances("my_trainset", {}, str(train_json), str(train_images))
if "my_testset" not in DatasetCatalog.list():
    register_coco_instances("my_testset", {}, str(test_json), str(test_images))

metadata = MetadataCatalog.get("my_trainset")
dataset_dicts = DatasetCatalog.get("my_trainset")
metadata_test = MetadataCatalog.get("my_testset")
dataset_dicts_test = DatasetCatalog.get("my_testset")

print(f"\n{metadata}\n \n{metadata_test}")

[02/07 17:08:38 d2.data.datasets.coco]: Loading /home/vasakjakub/emergence-detection/models/segmentation_model/annotations/train/result.json takes 2.20 seconds.
[02/07 17:08:38 d2.data.datasets.coco]: Loaded 61 images in COCO format from /home/vasakjakub/emergence-detection/models/segmentation_model/annotations/train/result.json
[02/07 17:08:40 d2.data.datasets.coco]: Loading /home/vasakjakub/emergence-detection/models/segmentation_model/annotations/test/result.json takes 1.91 seconds.
[02/07 17:08:40 d2.data.datasets.coco]: Loaded 12 images in COCO format from /home/vasakjakub/emergence-detection/models/segmentation_model/annotations/test/result.json

Metadata(name='my_trainset', json_file='/home/vasakjakub/emergence-detection/models/segmentation_model/annotations/train/result.json', image_root='/home/vasakjakub/emergence-detection/models/segmentation_model/annotations/train/images', evaluator_type='coco', thing_classes=['Clay'], thing_dataset_id_to_contiguous_id={1: 0})
 
Metadata(na

In [ ]:
# Check if the data is loaded correctly

for d in random.sample(dataset_dicts, 5):
    img = cv2.imread(d["file_name"])
    visualizer = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.5)
    vis = visualizer.draw_dataset_dict(d)
    plt.figure(figsize=(15, 13))
    plt.imshow(cv2.cvtColor((vis.get_image()[:, :, ::-1]), cv2.COLOR_BGR2RGB))
    plt.show()

## Training the model

- The get_cfg function is used to create a new configuration that holds default values for configurations. 
- The merge_from_file function is then used to merge the values from a YAML file that contains the pre-defined configurations for the Mask R-CNN model.
- The configuration object is then customized for the specific training task.
- THe directory for the output is created if it doesn't exist. The model is then trained using the DefaultTrainer class.
- On T4 GPU the training time is aproximatly 50 mins.

In [ ]:
from detectron2.engine import DefaultTrainer

# For model fine-tuning, adjust the model choice and the configuration if needed.
cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
)
cfg.DATASETS.TRAIN = ("my_trainset",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 4  # 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)
cfg.SOLVER.IMS_PER_BATCH = 8  # 4
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 1300
cfg.SOLVER.STEPS = []
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1

SEGMENTATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.OUTPUT_DIR = str(SEGMENTATION_OUTPUT_DIR)

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

[02/07 17:09:35 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

model_final_f10217.pkl: 178MB [00:01, 161MB/s]                             
Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (2, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (2,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (4, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (4,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.mask_head.predictor.weight' to the model due to 

[02/07 17:09:37 d2.engine.train_loop]: Starting training from iteration 0


/opt/conda/envs/emergence/lib/python3.10/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[02/07 17:10:48 d2.utils.events]:  eta: 0:51:50  iter: 19  total_loss: 3.543  loss_cls: 0.6003  loss_box_reg: 0.5556  loss_mask: 0.6925  loss_rpn_cls: 1.56  loss_rpn_loc: 0.1484    time: 3.2100  last_time: 2.4322  data_time: 1.0229  last_data_time: 0.0211   lr: 4.9953e-06  max_mem: 9571M


2025-02-07 17:10:51.570409: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-07 17:10:51.570508: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-07 17:10:51.747694: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-07 17:10:52.095630: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[02/07 17:11:47 d2.utils.events]:  eta: 0:51:04  iter: 39  total_loss: 2.194  loss_cls: 0.5969  loss_box_reg: 0.6116  loss_mask: 0.6804  loss_rpn_cls: 0.2064  loss_rpn_loc: 0.09782    time: 2.8195  last_time: 2.3177  data_time: 0.0944  last_data_time: 0.0447   lr: 9.9902e-06  max_mem: 9571M
[02/07 17:12:38 d2.utils.events]:  eta: 0:50:53  iter: 59  total_loss: 2.034  loss_cls: 0.5772  loss_box_reg: 0.6247  loss_mask: 0.6541  loss_rpn_cls: 0.09966  loss_rpn_loc: 0.08238    time: 2.7075  last_time: 2.3452  data_time: 0.0560  last_data_time: 0.0195   lr: 1.4985e-05  max_mem: 9571M
[02/07 17:13:29 d2.utils.events]:  eta: 0:50:24  iter: 79  total_loss: 1.912  loss_cls: 0.5482  loss_box_reg: 0.6194  loss_mask: 0.6187  loss_rpn_cls: 0.05561  loss_rpn_loc: 0.07689    time: 2.6597  last_time: 2.3980  data_time: 0.0496  last_data_time: 0.0284   lr: 1.998e-05  max_mem: 9573M
[02/07 17:14:21 d2.utils.events]:  eta: 0:50:00  iter: 99  total_loss: 1.825  loss_cls: 0.5213  loss_box_reg: 0.6118  loss_

## Display training curves using tensorboard
 - You could use tensorboard to show the learning curves, if you have installed it

In [ ]:
%reload_ext tensorboard
%load_ext tensorboard
%tensorboard --logdir output

## Test sample

- cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7 sets the threshold for the prediction score. Only regions with a score above this threshold will be considered in the final prediction.
- predictor = DefaultPredictor(cfg) creates a predictor object using the defined configuration. This object can be used to make predictions on new data.
- print(inference_on_dataset(trainer.model, test_loader, test_evaluator)) performs inference on the test dataset using the trained model and the test data loader, evaluates the results using the test evaluator, and prints the results

In [7]:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7
cfg.DATASETS.TEST = ("my_testset",)
predictor = DefaultPredictor(cfg)

[02/07 18:09:31 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /home/vasakjakub/emergence-detection//models/segmentation_model/output/model_final.pth ...


In [ ]:
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

outputs = cfg.OUTPUT_DIR

# Create evaluator with explicit arguments instead of full config
test_evaluator = COCOEvaluator(
    dataset_name="my_testset",
    tasks=("bbox", "segm"),  # specify which tasks to evaluate
    distributed=False,
    output_dir=outputs,  # specify your output directory explicitly
)

test_loader = build_detection_test_loader(cfg, "my_testset")  # type: ignore
print(inference_on_dataset(trainer.model, test_loader, test_evaluator))  # type: ignore

[02/07 18:14:08 d2.data.datasets.coco]: Loaded 12 images in COCO format from /home/vasakjakub/emergence-detection/models/segmentation_model/annotations/test/result.json
[02/07 18:14:08 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[02/07 18:14:08 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[02/07 18:14:08 d2.data.common]: Serializing 12 elements to byte tensors and concatenating them all ...
[02/07 18:14:08 d2.data.common]: Serialized dataset takes 0.50 MiB
[02/07 18:14:08 d2.evaluation.evaluator]: Start inference on 12 batches
[02/07 18:14:26 d2.evaluation.evaluator]: Inference done 11/12. Dataloading: 0.0014 s/iter. Inference: 0.2522 s/iter. Eval: 1.2339 s/iter. Total: 1.4875 s/iter. ETA=0:00:01
[02/07 18:14:27 d2.evaluation.evaluator]: Total inference time: 0:00:10.402966 (1.486138 s / iter per device, on 1 dev

## Visualize the predictions on the test sample

In [ ]:
from detectron2.utils.visualizer import ColorMode

for d in random.sample(dataset_dicts_test, 11):
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)
    v = Visualizer(
        im[:, :, ::-1], metadata=metadata_test, scale=0.8, instance_mode=ColorMode.IMAGE
    )
    v = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    plt.figure(figsize=(15, 13))
    plt.imshow(cv2.cvtColor(v.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
    plt.show()